<a href="https://colab.research.google.com/drive/1DaKn5yG88up_fE2FHlVluCZstUD5l7oF?usp=sharing" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import re
import numpy as np

In [2]:
df = pd.read_csv("/content/iproperty_raw_102k.csv")

# Data Cleaning

In [3]:
# Remove duplicate
initial_count = len(df)
df.drop_duplicates(subset=['listing_url'], keep='first', inplace=True)
print(f"Removed {initial_count - len(df)} duplicate rows.")

Removed 7498 duplicate rows.


In [4]:
# Initialize new columns to store extracted data
df['price_rm'] = np.nan
df['price_psf'] = np.nan
df['built_up_sqft'] = np.nan
df['land_area_sqft'] = np.nan
df['property_type'] = ""
df['furnishing'] = ""

In [5]:
for index, row in df.iterrows():
  text = str(row['raw_text'])

  # 1. Extract Total Price (Grabs the starting price if it's a range)
  price_match = re.search(r'RM\s*([\d,]+)', text)
  if price_match:
    df.at[index, 'price_rm'] = float(price_match.group(1).replace(',', ''))

  # 2. Extract Price Per Sqft
  psf_match = re.search(r'RM\s*([\d,.]+)\s*psf', text, flags=re.IGNORECASE)
  if psf_match:
    df.at[index, 'price_psf'] = float(psf_match.group(1).replace(',', ''))

  # 3. Extract Built-up / Floor Size
  floor_match = re.search(r'([\d,]+)\s*sqft(?:\s*\(floor\))?', text, flags=re.IGNORECASE)
  if floor_match:
    df.at[index, 'built_up_sqft'] = float(floor_match.group(1).replace(',', ''))

  # 4. Extract Land Size
  land_match = re.search(r'([\d,]+)\s*sqft\s*\(land\)', text, flags=re.IGNORECASE)
  if land_match:
    df.at[index, 'land_area_sqft'] = float(land_match.group(1).replace(',', ''))

  # 5. Extract Property Type
  if 'Bungalow' in text or 'Villa' in text:
    df.at[index, 'property_type'] = 'Bungalow/Villa'
  elif 'Condominium' in text or 'Serviced Residence' in text:
    df.at[index, 'property_type'] = 'Condominium'
  elif 'Apartment' in text or 'Flat' in text:
    df.at[index, 'property_type'] = 'Apartment/Flat'
  elif 'Terrace' in text or 'Terraced' in text or 'Link' in text:
    df.at[index, 'property_type'] = 'Terrace/Link'
  elif 'Semi-D' in text or 'Semi-Detached' in text:
    df.at[index, 'property_type'] = 'Semi-Detached'

  # 6. Extract Furnishing Status
  if 'Unfurnished' in text:
    df.at[index, 'furnishing'] = 'Unfurnished'
  elif 'Partly' in text or 'Partially' in text:
    df.at[index, 'furnishing'] = 'Partially Furnished'
  elif 'Fully' in text:
    df.at[index, 'furnishing'] = 'Fully Furnished'

Missing value

In [6]:
# Fill missing values with the median
df['price_rm'] = df['price_rm'].fillna(df['price_rm'].median())
df['price_psf'] = df['price_psf'].fillna(df['price_psf'].median())

In [7]:
# Fill missing categorical values with 'Unknown'
df['property_type'] = df['property_type'].replace("", "Unknown")
df['furnishing'] = df['furnishing'].replace("", "Unknown")

In [8]:
# Convert 'scraped_at' to DateTime object
df['scraped_at'] = df['scraped_at'].str.replace(r'\s+', ' ', regex=True).str.strip()
df['scraped_at'] = pd.to_datetime(df['scraped_at'], dayfirst=True, format='mixed', errors='coerce')

# Data Structure

In [9]:
final_columns = [
        'listing_url',
        'property_type',
        'price_rm',
        'price_psf',
        'built_up_sqft',
        'land_area_sqft',
        'furnishing',
        'scraped_at'
    ]

In [10]:
cleaned_df = df[final_columns]

In [11]:
cleaned_df = cleaned_df.astype({
        'price_rm': 'float64',
        'price_psf': 'float64',
        'property_type': 'string',
        'furnishing': 'string'
    })

In [12]:
output_filepath = '/content/iproperty_cleaned.csv'

In [13]:
# Export cleaned dataframe
cleaned_df.to_csv(output_filepath, index=False)
print(f"Dataset successfully saved to {output_filepath}")

Dataset successfully saved to /content/iproperty_cleaned.csv


In [14]:
display(cleaned_df.head(10))

,listing_url,property_type,price_rm,price_psf,built_up_sqft,land_area_sqft,furnishing,scraped_at
0,https://www.iproperty.com.my/property/iskandar...,Bungalow/Villa,4900000.0,482.00,7200.0,10166.0,Unfurnished,2026-06-21 20:50:17
1,https://www.iproperty.com.my/property/johor-ba...,Terrace/Link,848000.0,403.81,1600.0,2100.0,Unfurnished,2026-06-21 20:50:17
2,https://www.iproperty.com.my/property/kepong/l...,Terrace/Link,1550000.0,939.39,3600.0,1650.0,Fully Furnished,2026-06-21 20:50:17
3,https://www.iproperty.com.my/property/bayan-le...,Apartment/Flat,500000.0,476.19,1050.0,NaN,Partially Furnished,2026-06-21 20:50:17
4,https://www.iproperty.com.my/property/skudai/t...,Terrace/Link,720000.0,250.00,1980.0,2880.0,Partially Furnished,2026-06-21 20:50:17
5,https://www.iproperty.com.my/new-property/prop...,Terrace/Link,386800.0,292.15,1324.0,NaN,Unknown,2026-06-21 20:50:17
6,https://www.iproperty.com.my/property/tampoi/d...,Apartment/Flat,395000.0,390.32,1012.0,NaN,Unfurnished,2026-06-21 20:50:17
7,https://www.iproperty.com.my/property/iskandar...,Terrace/Link,820000.0,455.56,2238.0,1800.0,Unfurnished,2026-06-21 20:50:17
8,https://www.iproperty.com.my/property/kampung-...,Unknown,516800.0,1044.04,495.0,NaN,Partially Furnished,2026-06-21 20:50:17
9,https://www.iproperty.com.my/new-property/prop...,Terrace/Link,1010000.0,397.32,2542.0,NaN,Unknown,2026-06-21 20:50:17


In [15]:
print(f"NaNs in built_up_sqft: {cleaned_df['built_up_sqft'].isna().sum()}")
print(f"NaNs in land_area_sqft: {cleaned_df['land_area_sqft'].isna().sum()}")

NaNs in built_up_sqft: 2
NaNs in land_area_sqft: 49990


In [16]:
print(f"Number of rows in cleaned_df: {len(cleaned_df)}")

Number of rows in cleaned_df: 95004
